# Experiments

This notebook studies the interaction between RM3 query expansion and TAS-B neural re-ranking on the MS MARCO Passage dataset.

Pipelines:
1. BM25 (baseline)
2. BM25 + RM3 (query expansion)
3. BM25 + TAS-B (neural re-ranking)
4. BM25 + RM3 + TAS-B (combined)

Evaluation: TREC Deep Learning 2019 & 2020 queries with NDCG@10, MAP, and Recall@100.

## Setup

Import PyTerrier and initialize the Java backend.

Dependencies: `python-terrier`, `pyterrier_dr`, `pandas`.

In [1]:
import pyterrier as pt
import pandas as pd
from pyterrier_dr import TasB

if not pt.java.started():
    pt.java.init()

The operation couldn’t be completed. Unable to locate a Java Runtime.
Please visit http://www.java.com for information on installing Java.



Exception: You must install Java for Mac OSX

## Load dataset

Load the MS MARCO Passage dataset. We use two dataset handles:
- `dataset` (RemoteDataset) for the pre-built index
- `irds_dataset` (IRDSDataset) for document text lookup (needed for TAS-B re-ranking)

Topics and qrels come from the TREC DL 2019 and 2020 judged query sets.

In [ ]:
dataset = pt.get_dataset("msmarco_passage")
irds_dataset = pt.get_dataset("irds:msmarco-passage")

dl19 = pt.get_dataset("irds:msmarco-passage/trec-dl-2019/judged")
dl20 = pt.get_dataset("irds:msmarco-passage/trec-dl-2020/judged")

topics_19 = dl19.get_topics()
qrels_19 = dl19.get_qrels()

topics_20 = dl20.get_topics()
qrels_20 = dl20.get_qrels()

print(f"DL 2019 — topics: {len(topics_19)}, qrels: {len(qrels_19)}")
print(f"DL 2020 — topics: {len(topics_20)}, qrels: {len(qrels_20)}")

## Load index

Load the pre-built Terrier index for MS MARCO Passage (downloaded automatically on first run).

In [ ]:
index = pt.IndexFactory.of(dataset.get_index(variant="terrier_stemmed"))

In [ ]:
print(index.getCollectionStatistics().toString())

Number of documents: 11429
Number of terms: 7756
Number of postings: 224573
Number of fields: 1
Number of tokens: 271581
Field names: [text]
Positions:   false



## BM25 baseline

Create the first retrieval pipeline.

In [ ]:
bm25 = pt.terrier.Retriever(index, wmodel="BM25", num_results=1000)

## Evaluate BM25

Run the baseline and inspect the first metrics.

In [ ]:
pt.Experiment(
    [bm25],
    topics_19,
    qrels_19,
    eval_metrics=["ndcg_cut_10", "map", "recall_100"],
    names=["BM25"]
)

## Add RM3

Create the query expansion pipeline using RM3 pseudo-relevance feedback.

In [ ]:
rm3 = pt.rewrite.RM3(index, fb_terms=10)
bm25_rm3 = bm25 >> rm3 >> bm25

In [ ]:
pt.Experiment(
    [bm25, bm25_rm3],
    topics_19,
    qrels_19,
    eval_metrics=["ndcg_cut_10", "map", "recall_100"],
    names=["BM25", "BM25+RM3"]
)

In [ ]:
rm3_results = []

for fb in [5, 10, 20, 30, 50]:
    rm3_var = pt.rewrite.RM3(index, fb_terms=fb)
    pipe = bm25 >> rm3_var >> bm25

    exp = pt.Experiment(
        [pipe],
        topics_19,
        qrels_19,
        eval_metrics=["ndcg_cut_10", "map", "recall_100"],
        names=[f"BM25+RM3(fb={fb})"]
    )
    rm3_results.append(exp)

pd.concat(rm3_results, ignore_index=True)

## Add TAS-B

Create the neural re-ranking pipeline. BM25 retrieves top-100 candidates, then TAS-B re-ranks them using semantic similarity.

In [ ]:
tasb = TasB.dot()
get_text = pt.text.get_text(irds_dataset, "text")

bm25_tasb = bm25 % 100 >> get_text >> tasb

pt.Experiment(
    [bm25, bm25_tasb],
    topics_19,
    qrels_19,
    eval_metrics=["ndcg_cut_10", "map", "recall_100"],
    names=["BM25", "BM25+TAS-B"]
)

## Full pipeline: BM25 + RM3 + TAS-B

Combine query expansion and neural re-ranking. BM25 retrieves, RM3 expands the query, BM25 re-retrieves with the expanded query, then TAS-B re-ranks the top 100 using the **original** query (since TAS-B is trained on natural language, not weighted term expansions).

In [ ]:
# Reset query to original after RM3 expansion so TAS-B sees natural language.
# Use pt.apply.generic (not pt.apply.query, which overwrites query_0 before we read it).
reset_query = pt.apply.generic(lambda df: df.assign(query=df["query_0"]))

bm25_rm3_tasb = bm25 >> rm3 >> bm25 % 100 >> reset_query >> get_text >> tasb

pt.Experiment(
    [bm25, bm25_rm3, bm25_tasb, bm25_rm3_tasb],
    topics_19,
    qrels_19,
    eval_metrics=["ndcg_cut_10", "map", "recall_100"],
    names=["BM25", "BM25+RM3", "BM25+TAS-B", "BM25+RM3+TAS-B"]
)

## Compare pipelines

Evaluate all four pipelines on both TREC DL 2019 and 2020 to verify consistency of findings.

In [ ]:
pipelines = [bm25, bm25_rm3, bm25_tasb, bm25_rm3_tasb]
names = ["BM25", "BM25+RM3", "BM25+TAS-B", "BM25+RM3+TAS-B"]

print("=== TREC DL 2019 ===")
dl19 = pt.Experiment(
    pipelines, topics_19, qrels_19,
    eval_metrics=["ndcg_cut_10", "map", "recall_100"],
    names=names
)
display(dl19)

print("\n=== TREC DL 2020 ===")
dl20 = pt.Experiment(
    pipelines, topics_20, qrels_20,
    eval_metrics=["ndcg_cut_10", "map", "recall_100"],
    names=names
)
display(dl20)

## RM3 parameter study with TAS-B

Vary the number of expansion terms (`fb_terms`) and measure the effect on TAS-B re-ranking performance. This directly addresses the research question: does the amount of query expansion help or hurt neural re-ranking?

In [ ]:
fb_values = [5, 10, 20, 30, 50]
param_results_19 = []
param_results_20 = []

for fb in fb_values:
    rm3_var = pt.rewrite.RM3(index, fb_terms=fb)
    pipe = bm25 >> rm3_var >> bm25 % 100 >> reset_query >> get_text >> tasb

    exp_19 = pt.Experiment(
        [pipe], topics_19, qrels_19,
        eval_metrics=["ndcg_cut_10", "map", "recall_100"],
        names=[f"BM25+RM3(fb={fb})+TAS-B"]
    )
    param_results_19.append(exp_19)

    exp_20 = pt.Experiment(
        [pipe], topics_20, qrels_20,
        eval_metrics=["ndcg_cut_10", "map", "recall_100"],
        names=[f"BM25+RM3(fb={fb})+TAS-B"]
    )
    param_results_20.append(exp_20)

# Include no-expansion baseline for reference
baseline_19 = pt.Experiment(
    [bm25_tasb], topics_19, qrels_19,
    eval_metrics=["ndcg_cut_10", "map", "recall_100"],
    names=["BM25+TAS-B (no expansion)"]
)
baseline_20 = pt.Experiment(
    [bm25_tasb], topics_20, qrels_20,
    eval_metrics=["ndcg_cut_10", "map", "recall_100"],
    names=["BM25+TAS-B (no expansion)"]
)

print("=== DL 2019: Effect of fb_terms on TAS-B re-ranking ===")
display(pd.concat([baseline_19] + param_results_19, ignore_index=True))

print("\n=== DL 2020: Effect of fb_terms on TAS-B re-ranking ===")
display(pd.concat([baseline_20] + param_results_20, ignore_index=True))